<a href="https://colab.research.google.com/github/chuy-zip/3D_PROJECT/blob/main/proyecto3_paralela.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [214]:
!git clone https://github.com/eunicean/Proyecto3-Paralela.git

fatal: destination path 'Proyecto3-Paralela' already exists and is not an empty directory.


In [215]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [216]:
# Detect selected GPU and its NVIDA architecture:
import subprocess
gpu_info = subprocess.getoutput("nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader,nounits")
if "not found" in gpu_info.lower(): raise RuntimeError("Error: No GPU found. Please select a GPU runtime environment.")
gpu_name, compute_cap = map(str.strip, gpu_info.split(','))
gpu_arch = f"sm_{compute_cap.replace('.', '')}"

print(f"{'GPU Name':<15}: {gpu_name}")
print(f"{'Architecture':<15}: {gpu_arch}")

GPU Name       : Tesla T4
Architecture   : sm_75


In [217]:
%%writefile pgm.h
#ifndef PGM_H
#define PGM_H

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

class PGMImage {
public:
    int x_dim, y_dim;
    unsigned char *pixels;

    PGMImage(const char *filename) {
        FILE *file = fopen(filename, "rb");
        if (!file) {
            printf("Error: No se pudo abrir el archivo %s\n", filename);
            x_dim = y_dim = 0;
            pixels = NULL;
            return;
        }

        char magic[3];
        fscanf(file, "%2s\n", magic);

        // Saltar comentarios
        char c = getc(file);
        while (c == '#') {
            while (getc(file) != '\n');
            c = getc(file);
        }
        ungetc(c, file);

        fscanf(file, "%d %d\n", &x_dim, &y_dim);
        int maxVal;
        fscanf(file, "%d\n", &maxVal);

        pixels = new unsigned char[x_dim * y_dim];
        fread(pixels, sizeof(unsigned char), x_dim * y_dim, file);
        fclose(file);
    }

    ~PGMImage() {
        if (pixels) delete[] pixels;
    }
};

#endif

Overwriting pgm.h


In [218]:
%%writefile houghBase.cu
/*
 ============================================================================
 Author        : G. Barlas
 Version       : 1.0
 Last modified : December 2014
 License       : Released under the GNU GPL 3.0
 Description   :
 To build use  : make
 ============================================================================
 */

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda.h>
#include <string.h>
#include "pgm.h"

const int degreeInc = 2;
const int degreeBins = 180 / degreeInc;
const int rBins = 100;
const float radInc = degreeInc * M_PI / 180;
//*****************************************************************
// The CPU function returns a pointer to the accummulator
void CPU_HoughTran (unsigned char *pic, int w, int h, int **acc)
{
  float rMax = sqrt (1.0 * w * w + 1.0 * h * h) / 2;
  *acc = new int[rBins * degreeBins];
  memset (*acc, 0, sizeof (int) * rBins * degreeBins);
  int xCent = w / 2;
  int yCent = h / 2;
  float rScale = 2 * rMax / rBins;

  printf("rMax CPU: %.6f\n", rMax);
  printf("rScale CPU: %.6f\n", rScale);

  for (int i = 0; i < w; i++)
    for (int j = 0; j < h; j++)
      {
        int idx = j * w + i;
        if (pic[idx] > 0)
          {
            int xCoord = i - xCent;
            int yCoord = yCent - j;
            float theta = 0;
            for (int tIdx = 0; tIdx < degreeBins; tIdx++)
              {
                float r = xCoord * cos (theta) + yCoord * sin (theta);
                // cambio para usar floorf para consistencia con GPU
                float normalized_r = (r + rMax) / rScale;
                int rIdx = (int)floorf(normalized_r);
                // cambio para Verificar límites como en GPU
                if (rIdx >= 0 && rIdx < rBins) {
                  (*acc)[rIdx * degreeBins + tIdx]++;
                }
                theta += radInc;
              }
          }
      }
}

//*****************************************************************
// TODO usar memoria constante para la tabla de senos y cosenos
// inicializarlo en main y pasarlo al device
//__constant__ float d_Cos[degreeBins];
//__constant__ float d_Sin[degreeBins];

//*****************************************************************
//TODO Kernel memoria compartida
// __global__ void GPU_HoughTranShared(...)
// {
//   //TODO
// }
//TODO Kernel memoria Constante
// __global__ void GPU_HoughTranConst(...)
// {
//   //TODO
// }

// GPU kernel. One thread per image pixel is spawned.
// The accummulator memory needs to be allocated by the host in global memory
__global__ void GPU_HoughTran (unsigned char *pic, int w, int h, int *acc, float rMax, float rScale, float *d_Cos, float *d_Sin)
{
  int gloID = blockIdx.x * blockDim.x + threadIdx.x;
  if (gloID >= w * h) return;

  int xCent = w / 2;
  int yCent = h / 2;

  int i = gloID % w;
  int j = gloID / w;

  int xCoord = i - xCent;
  int yCoord = yCent - j;

  if (pic[gloID] > 0)
  {
    for (int tIdx = 0; tIdx < degreeBins; tIdx++)
    {
      float r = xCoord * d_Cos[tIdx] + yCoord * d_Sin[tIdx];

      // Cálculo más preciso de rIdx - MANTENER ESTO
      float normalized_r = (r + rMax) / rScale;
      int rIdx = (int)floorf(normalized_r);

      // Verificación más estricta - MANTENER ESTO
      if (rIdx >= 0 && rIdx < rBins) {
        int acc_index = rIdx * degreeBins + tIdx;
        atomicAdd(acc + acc_index, 1);
      }
    }
  }
}

//*****************************************************************
int main (int argc, char **argv)
{
  int i;

  PGMImage inImg (argv[1]);

  int *cpuht;
  int w = inImg.x_dim;
  int h = inImg.y_dim;

  // verificar pixeles
  printf("Dimensiones imagen: %d x %d\n", w, h);

  float* d_Cos;
  float* d_Sin;

  cudaMalloc ((void **) &d_Cos, sizeof (float) * degreeBins);
  cudaMalloc ((void **) &d_Sin, sizeof (float) * degreeBins);

  // CPU calculation
  CPU_HoughTran(inImg.pixels, w, h, &cpuht);

  // pre-compute values to be stored
  float *pcCos = (float *) malloc (sizeof (float) * degreeBins);
  float *pcSin = (float *) malloc (sizeof (float) * degreeBins);
  float rad = 0;
  for (i = 0; i < degreeBins; i++)
  {
    pcCos[i] = cos (rad);
    pcSin[i] = sin (rad);
    rad += radInc;
  }

  float rMax = sqrt (1.0 * w * w + 1.0 * h * h) / 2;
  float rScale = 2 * rMax / rBins;

  // TODO eventualmente volver memoria global
  cudaMemcpy(d_Cos, pcCos, sizeof (float) * degreeBins, cudaMemcpyHostToDevice);
  cudaMemcpy(d_Sin, pcSin, sizeof (float) * degreeBins, cudaMemcpyHostToDevice);

  // setup and copy data from host to device
  unsigned char *d_in, *h_in;
  int *d_hough, *h_hough;

  h_in = inImg.pixels;

  h_hough = (int *) malloc (degreeBins * rBins * sizeof (int));

  cudaMalloc ((void **) &d_in, sizeof (unsigned char) * w * h);
  cudaMalloc ((void **) &d_hough, sizeof (int) * degreeBins * rBins);
  cudaMemcpy (d_in, h_in, sizeof (unsigned char) * w * h, cudaMemcpyHostToDevice);
  cudaMemset (d_hough, 0, sizeof (int) * degreeBins * rBins);

  // execution configuration uses a 1-D grid of 1-D blocks, each made of 256 threads
  //1 thread por pixel
  int blockNum = (w * h + 255) / 256;

  // Record elapsed time using CUDA events - Dan
  cudaEvent_t start, stop;
  cudaEventCreate (&start);
  cudaEventCreate (&stop);

  cudaEventRecord(start);
  GPU_HoughTran <<< blockNum, 256 >>> (d_in, w, h, d_hough, rMax, rScale, d_Cos, d_Sin);
  cudaEventRecord(stop);

  cudaEventSynchronize(stop);

  // CAMBIO para Verificar errores de CUDA después del kernel
  cudaError_t err = cudaGetLastError();
  if (err != cudaSuccess) {
    printf("Error CUDA después del kernel: %s\n", cudaGetErrorString(err));
  }

  float elapsedTime = 0;
  cudaEventElapsedTime(&elapsedTime, start, stop);

  // get results from device
  cudaMemcpy (h_hough, d_hough, sizeof (int) * degreeBins * rBins, cudaMemcpyDeviceToHost);

  // compare CPU and GPU results
  printf("Verificando resultados...\n");
  int errors = 0;
  for (i = 0; i < degreeBins * rBins; i++)
  {
    if (cpuht[i] != h_hough[i]) {
      printf ("Diferencia en [%i]: CPU=%i, GPU=%i\n", i, cpuht[i], h_hough[i]);
      errors++;
      if (errors > 30) break;
    }
  }

  if (errors == 0) {
    printf("Resultados correctos! CPU == GPU\n");
  } else {
    printf("Cantidad de errores: %d\n", errors);
  }

  printf("Tiempo GPU: %.3f ms\n", elapsedTime);
  printf("Done!\n");

  // Liberar memoria
  free(pcCos);
  free(pcSin);
  free(cpuht);
  free(h_in);
  free(h_hough);

  cudaFree(d_Cos);
  cudaFree(d_Sin);
  cudaFree(d_in);
  cudaFree(d_hough);

  return 0;
}

Overwriting houghBase.cu


In [219]:
!nvcc -arch=sm_70 houghBase.cu -o hough

In [220]:
!./hough /content/Proyecto3-Paralela/runway.pgm

Dimensiones imagen: 800 x 600
rMax CPU: 500.000000
rScale CPU: 10.000000
Verificando resultados...
Diferencia en [1803]: CPU=1446, GPU=1445
Diferencia en [1893]: CPU=1506, GPU=1507
Diferencia en [5931]: CPU=1653, GPU=1654
Diferencia en [6021]: CPU=1816, GPU=1815
Diferencia en [6104]: CPU=1642, GPU=1641
Diferencia en [6194]: CPU=1586, GPU=1587
Cantidad de errores: 6
Tiempo GPU: 1.632 ms
Done!
